# Диаризация аудио с NVIDIA NeMo Clustering + MSDD

### Установка зависимостей

После первой установки выбрать Среда выполнения -> Перезапустить сеанс, затем продолжить с ячейки инициализации ниже, не запуская установку повторно.

In [ ]:
%pip install -q Cython packaging "nemo_toolkit[asr]==2.7.0" memory_profiler


### Инициализация девайса и библиотек

После перезапуска начать отсюда.

In [ ]:
import json
import re
import subprocess
from pathlib import Path
from time import perf_counter

import pandas as pd
from memory_profiler import memory_usage
import torch
from google.colab import files
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден. Будет использован CPU.")


### Загрузка аудиофайла

Выбрать один аудиофайл или медиаконтейнер с аудиодорожкой из числа поддерживаемых FFmpeg.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Файл не загружен.")
if len(uploaded) != 1:
    raise ValueError("Загружено несколько файлов, вместо одного.")

audio_name = next(iter(uploaded))
audio_path = Path(audio_name)
allowed_extensions = {
    ".3g2", ".3gp", ".aac", ".ac3", ".aif", ".aifc", ".aiff",
    ".amr", ".ape", ".au", ".avi", ".awb", ".caf", ".dts",
    ".eac3", ".flac", ".flv", ".gsm", ".m2ts", ".m4a", ".m4b",
    ".mka", ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpc",
    ".mpeg", ".mpg", ".mts", ".oga", ".ogg", ".opus", ".ra",
    ".rm", ".snd", ".spx", ".tak", ".ts", ".tta", ".wav",
    ".wave", ".webm", ".wma", ".wv",
}
if audio_path.suffix.lower() not in allowed_extensions:
    raise ValueError(
        f"Неподдерживаемый формат {audio_path.suffix or 'без расширения'}. "
        "Выберите аудиофайл или медиаконтейнер с поддерживаемой аудиодорожкой."
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise ValueError("Загруженный файл отсутствует или пуст.")
print(f"Загружен файл: {audio_name} ({audio_path.stat().st_size / 1024 / 1024:.2f} МБ)")

### Подготовка аудио

In [ ]:
prepared_path = Path("/tmp/prepared_audio.wav")
command = [
    "ffmpeg", "-v", "error", "-y", "-i", str(audio_path),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(prepared_path),
]
try:
    conversion = subprocess.run(command, capture_output=True, text=True, check=False)
except FileNotFoundError as exc:
    raise RuntimeError("FFmpeg not found.") from exc

if conversion.returncode != 0 or not prepared_path.is_file() or prepared_path.stat().st_size <= 44:
    details = (conversion.stderr or "FFmpeg error").strip()[-1000:]
    raise RuntimeError(
        "Не удалось прочитать или преобразовать аудио."
        f"Сообщение FFmpeg: {details}"
    )
print(f"Аудио подготовлено: {prepared_path} (mono, 16 кГц, WAV)")

### Загрузка моделей

Используются открытые модели NeMo: diar_msdd_telephonic и vad_multilingual_marblenet.

In [ ]:
from nemo.collections.asr.models import NeuralDiarizer

MODEL_ID = "diar_msdd_telephonic"
VAD_MODEL_ID = "vad_multilingual_marblenet"
try:
    diar_model = NeuralDiarizer.from_pretrained(
        model_name=MODEL_ID,
        vad_model_name=VAD_MODEL_ID,
        map_location=str(DEVICE),
        verbose=False,
    )
    diar_model.to(DEVICE)
    diar_model.eval()
except Exception as exc:
    message = str(exc)
    raise RuntimeError(f"Не удалось загрузить модели: {message}") from exc
print(f"Модели загружены на {DEVICE}.")

### Настройка VAD

In [ ]:
vad_parameters = diar_model._cfg.diarizer.vad.parameters

vad_parameters.window_length_in_sec = 0.15
vad_parameters.shift_length_in_sec = 0.01
vad_parameters.smoothing = "median"
vad_parameters.overlap = 0.5

vad_parameters.onset = 0.10
vad_parameters.offset = 0.10
vad_parameters.pad_onset = 0.10
vad_parameters.pad_offset = 0.0
vad_parameters.min_duration_on = 0.0
vad_parameters.min_duration_off = 0.20
vad_parameters.filter_speech_first = True

### Выполнение диаризации

Если число участников известно, задать `NUM_SPEAKERS` положительным целым числом, `OVERLAP_INFER_SPK_LIMIT` автоматически устанавливается на единицу больше. Чтобы включить автоматическое определение, задать `NUM_SPEAKERS = None` и указать `OVERLAP_INFER_SPK_LIMIT` вручную. При `USE_ADAPTIVE_THRESHOLD = False` значение `MSDD_THRESHOLD` используется без автоматического повышения в зависимости от числа говорящих.

In [ ]:
NUM_SPEAKERS = 5
OVERLAP_INFER_SPK_LIMIT = NUM_SPEAKERS + 1
MSDD_THRESHOLD = 0.7
USE_ADAPTIVE_THRESHOLD = False
BATCH_SIZE = 64

if NUM_SPEAKERS is not None and (
    isinstance(NUM_SPEAKERS, bool)
    or not isinstance(NUM_SPEAKERS, int)
    or NUM_SPEAKERS < 1
):
    raise ValueError(
        "NUM_SPEAKERS должен быть положительным целым числом или None."
    )
if (
    isinstance(OVERLAP_INFER_SPK_LIMIT, bool)
    or not isinstance(OVERLAP_INFER_SPK_LIMIT, int)
    or OVERLAP_INFER_SPK_LIMIT < 1
):
    raise ValueError(
        "OVERLAP_INFER_SPK_LIMIT должен быть положительным целым числом."
    )
if not isinstance(MSDD_THRESHOLD, (int, float)) or isinstance(MSDD_THRESHOLD, bool) or not 0 < MSDD_THRESHOLD <= 1:
    raise ValueError("MSDD_THRESHOLD должен находиться в диапазоне (0, 1].")
if not isinstance(USE_ADAPTIVE_THRESHOLD, bool):
    raise ValueError("USE_ADAPTIVE_THRESHOLD должен быть True или False.")

diar_model._cfg.diarizer.msdd_model.parameters.sigmoid_threshold = (
    float(MSDD_THRESHOLD),
)
diar_model._cfg.diarizer.msdd_model.parameters.use_adaptive_thres = (
    USE_ADAPTIVE_THRESHOLD
)
diar_model._cfg.diarizer.msdd_model.parameters.overlap_infer_spk_limit = (
    OVERLAP_INFER_SPK_LIMIT
)
diar_model.use_adaptive_thres = USE_ADAPTIVE_THRESHOLD
diar_model.overlap_infer_spk_limit = OVERLAP_INFER_SPK_LIMIT

def run_diarization():
    with torch.inference_mode():
        return diar_model(
            audio_filepath=str(prepared_path.resolve()),
            batch_size=BATCH_SIZE,
            num_workers=1,
            num_speakers=NUM_SPEAKERS,
            verbose=False,
        )

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
diarization_started_at = perf_counter()

try:
    if DEVICE.type == "cuda":
        output = run_diarization()
    else:
        memory_values, output = memory_usage(
            run_diarization,
            retval=True,
            interval=0.1,
        )
except (torch.cuda.OutOfMemoryError, MemoryError) as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "Во время диаризации закончилась оперативная или видеопамять. "
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Диаризация завершилась ошибкой: {exc}."
    ) from exc

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    used_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
    memory_measurement_name = "Пиковое использование VRAM"
else:
    used_memory_gib = max(memory_values) / 1024
    memory_measurement_name = "Пиковое использование RAM"
diarization_elapsed_seconds = perf_counter() - diarization_started_at

if not hasattr(output, "itertracks"):
    raise RuntimeError("NeMo вернул результат неизвестного формата.")

def normalize_speaker(value):
    match = re.search(r"(\d+)$", str(value))
    if not match:
        raise ValueError(
            f"Не удалось определить номер говорящего из {value!r}."
        )
    return f"SPEAKER_{int(match.group(1)):02d}"

segments = []
for turn, _, speaker in output.itertracks(yield_label=True):
    start = float(turn.start)
    end = float(turn.end)
    if end > start:
        segments.append(
            {
                "start": round(start, 3),
                "end": round(end, 3),
                "speaker": normalize_speaker(speaker),
            }
        )
segments.sort(key=lambda item: (item["start"], item["end"], item["speaker"]))
if not segments:
    raise RuntimeError("В записи не обнаружена речь.")
print(f"Диаризация завершена. Найдено сегментов: {len(segments)}")

### Просмотр и скачивание временной разметки

Сегменты выводятся в таблице в минутах, полученная временная разметка сохраняется в json (в секундах).

In [ ]:
def format_minutes(seconds):
    minutes, remaining_seconds = divmod(float(seconds), 60)
    return f"{int(minutes):02d}:{remaining_seconds:06.3f}"

table = pd.DataFrame(
    [
        (format_minutes(item["start"]), format_minutes(item["end"]), item["speaker"])
        for item in segments
    ],
    columns=["Начало", "Окончание", "Говорящий"],
)
print(
    f"Время диаризации: {format_minutes(diarization_elapsed_seconds)} "
    f"({diarization_elapsed_seconds:.3f} с)"
)
print(f"{memory_measurement_name}: {used_memory_gib:.3f} GiB")
with pd.option_context("display.max_rows", None):
    display(table)

result = {"audio_file": audio_name, "segments": segments}
result_path = Path(audio_name).with_suffix(".json")
with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Результат сохранён: {result_path.resolve()}")
files.download(str(result_path))